# Лабораторная работа 1. Регрессия и метод главных компонент

**Датасет:** Bike Sharing Dataset (Kaggle, `hour.csv`) — почасовая аренда велосипедов.

**Цель работы:** изучение основ линейной регрессии, построение простейших моделей регрессии, проведение обучения модели на реальных данных и оценка её качества.

## 1. Постановка задачи

1. Загрузить датасет из репозитория (например, kaggle.com или аналогичных платформ).
2. Подготовить данные: провести первичный анализ, визуализировать распределение признаков и целевой переменной.
3. Провести предобработку данных: удалить пропущенные значения, закодировать категориальные переменные.
4. Построить матрицу корреляций. Сделать выводы о наличии мультиколлинеарности с помощью VIF-коэффициента.
5. Построить линейную, Lasso и Ridge регрессии. Разделить данные на тренировочную и тестовую выборки в соотношении 80/20. Использовать кросс-валидацию. Оценить качество моделей по RMSE, R² и MAPE.
6. Устранить мультиколлинеарность и снизить размерность признаков с помощью метода главных компонент (PCA). Перед PCA провести стандартизацию данных.
7. Повторить построение моделей на главных компонентах и сравнить результаты с моделями на исходных признаках.

## 2. Описание датасета

В работе используется датасет **Bike Sharing Dataset**, содержащий почасовую информацию об аренде велосипедов.

**Целевая переменная:**

- `cnt` — общее количество аренд велосипедов за час.

**Используемые признаки:**

- `hr` — час суток от 0 до 23;
- `holiday` — признак праздничного дня;
- `weekday` — день недели;
- `weathersit` — категория погодных условий;
- `temp` — нормализованная температура;
- `hum` — нормализованная влажность;
- `windspeed` — нормализованная скорость ветра.

Признаки `casual` и `registered` не используются в моделях, поскольку они непосредственно входят в состав целевой переменной `cnt`: `cnt = casual + registered`. Их использование привело бы к утечке информации о целевой переменной.

Признаки `instant`, `dteday`, `yr`, `season`, `mnth` и `workingday` также не используются в итоговой модели. Признак `atemp` был удалён после анализа мультиколлинеарности, так как он имеет очень сильную связь с `temp`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.decomposition import PCA
from sklearn.metrics import (
    mean_squared_error,
    r2_score,
    mean_absolute_percentage_error
)
from statsmodels.stats.outliers_influence import variance_inflation_factor

import statsmodels.api as sm
from statsmodels.stats.diagnostic import het_breuschpagan

import warnings
warnings.filterwarnings('ignore')

В начале импортируются библиотеки для обработки данных, построения графиков, обучения регрессионных моделей, стандартизации, PCA и оценки качества моделей.

Основными объектами дальнейшей работы будут `df` — таблица с данными, `X` — матрица признаков и `y` — целевая переменная `cnt`.

In [ ]:
df = pd.read_csv('hour.csv')
df['dteday'] = pd.to_datetime(df['dteday'])

print('Первые 5 строк:')
print(df.head())

print('\nРазмер исходного датасета:')
print(df.shape)

print('\nИнформация о датасете:')
print(df.info())

print('\nНазвания столбцов:')
print(df.columns.tolist())

print('\nКоличество пропущенных значений:')
print(df.isnull().sum())

print('\nОсновная статистика:')
print(df.describe())

На первом этапе выполнен первичный анализ исходного датасета: проверены его размер, названия столбцов, типы данных, наличие пропущенных значений и основные статистические характеристики.

In [ ]:
df = df[
    (df['dteday'].dt.year == 2011) &
    (df['dteday'].dt.month == 5)
].copy()

print('Размер датасета за май 2011 года:')
print(df.shape)

print('\nПериод:')
print(df['dteday'].min(), '—', df['dteday'].max())

print('\nПервые 5 строк:')
print(df.head())

print('\nКоличество пропущенных значений:')
print(df.isnull().sum())

print('\nОсновная статистика:')
print(df.describe())

Для исследования выбран полный месяц — май 2011 года. После фильтрации получено **744 наблюдения** за период с 1 по 31 мая 2011 года.

В выбранной выборке пропущенные значения отсутствуют. Поэтому удаление строк с пропусками дополнительно не требуется.

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(df['cnt'], bins=30)
plt.title('Распределение количества аренд велосипедов')
plt.xlabel('Количество аренд за час (cnt)')
plt.ylabel('Частота')
plt.grid(alpha=0.3)
plt.show()

plt.figure(figsize=(8, 5))
plt.hist(df['temp'], bins=30)
plt.title('Распределение температуры')
plt.xlabel('Нормализованная температура (temp)')
plt.ylabel('Частота')
plt.grid(alpha=0.3)
plt.show()

plt.figure(figsize=(8, 5))
plt.hist(df['hum'], bins=30)
plt.title('Распределение влажности')
plt.xlabel('Нормализованная влажность (hum)')
plt.ylabel('Частота')
plt.grid(alpha=0.3)
plt.show()

plt.figure(figsize=(8, 5))
plt.hist(df['windspeed'], bins=30)
plt.title('Распределение скорости ветра')
plt.xlabel('Нормализованная скорость ветра (windspeed)')
plt.ylabel('Частота')
plt.grid(alpha=0.3)
plt.show()

На графиках представлены распределения целевой переменной `cnt`, температуры `temp`, влажности `hum` и скорости ветра `windspeed`.

Распределение количества аренд неоднородно, что связано с различиями между часами суток и условиями окружающей среды. Распределения числовых признаков показывают диапазоны значений, которые далее используются при построении моделей.

## 3. Подготовка данных

Сначала удаляются признаки, которые не будут использоваться в модели.

Корреляционный анализ будет проведён **до One-Hot Encoding**. Это сделано для того, чтобы матрица корреляций оставалась компактной и понятной. После этого `hr`, `weekday` и `weathersit` будут закодированы отдельно для использования в регрессионных моделях.

In [ ]:
df_analysis = df.drop(columns=[
    'instant',
    'dteday',
    'yr',
    'season',
    'mnth',
    'casual',
    'registered',
    'workingday',
    'atemp'
]).copy()

print('Данные после удаления ненужных признаков:')
print(df_analysis.head())

print('\nРазмер подготовленного датасета:')
print(df_analysis.shape)

print('\nОставшиеся столбцы:')
print(df_analysis.columns.tolist())

print('\nПропущенные значения:')
print(df_analysis.isnull().sum())

После удаления ненужных признаков остались следующие переменные: `hr`, `holiday`, `weekday`, `weathersit`, `temp`, `hum`, `windspeed` и `cnt`.

`cnt` используется как целевая переменная, остальные семь переменных рассматриваются как исходные признаки.

## 4. Корреляционный анализ

Для оценки линейных зависимостей построена корреляционная матрица Пирсона.

На этом этапе `hr`, `weekday` и `weathersit` ещё не кодируются. Это позволяет получить компактную матрицу и увидеть связь целевой переменной с исходными признаками.

Важно учитывать, что корреляция Пирсона показывает именно линейную связь. Поэтому небольшая корреляция `hr` с `cnt` не означает отсутствия влияния часа суток: зависимость аренды от времени может быть нелинейной.

In [ ]:
correlation_matrix = df_analysis.corr()

print('Корреляционная матрица:')
print(correlation_matrix)

plt.figure(figsize=(10, 8))
plt.imshow(
    correlation_matrix,
    cmap='coolwarm',
    aspect='auto',
    vmin=-1,
    vmax=1
)
plt.colorbar(label='Коэффициент корреляции')
plt.xticks(
    range(len(correlation_matrix.columns)),
    correlation_matrix.columns,
    rotation=45,
    ha='right'
)
plt.yticks(
    range(len(correlation_matrix.columns)),
    correlation_matrix.columns
)
plt.title('Корреляционная матрица признаков')
plt.tight_layout()
plt.show()

target_correlations = correlation_matrix['cnt'].sort_values(ascending=False)
print('\nКорреляция признаков с целевой переменной cnt:')
print(target_correlations)

Наиболее заметная обратная линейная связь с `cnt` наблюдается у `hum`: коэффициент составляет примерно **−0.50**. Для `temp` наблюдается прямая связь около **0.43**, а для `windspeed` — около **0.23**.

Таким образом, линейные связи с целевой переменной присутствуют, однако коэффициенты не достигают значений, близких к 1 по модулю. Это означает, что одной линейной зависимости недостаточно для полного объяснения изменения количества аренд.

Особенно важно учитывать `hr`: количество аренд может сильно различаться в течение суток, но эта зависимость не является простой линейной функцией часа. Поэтому далее `hr` будет преобразован с помощью One-Hot Encoding.

In [ ]:
df = df_analysis.copy()

df = pd.get_dummies(
    df,
    columns=['hr', 'weekday', 'weathersit'],
    drop_first=True,
    dtype=int
)

print('Данные после One-Hot Encoding:')
print(df.head())

print('\nРазмер датасета после кодирования:')
print(df.shape)

print('\nОставшиеся столбцы:')
print(df.columns.tolist())

y = df['cnt']
X = df.drop(columns=['cnt'])

print('\nРазмер X:', X.shape)
print('Размер y:', y.shape)

После One-Hot Encoding категориальные признаки `hr`, `weekday` и `weathersit` были преобразованы в бинарные переменные.

В результате получено **35 признаков**. Матрица `X` содержит эти признаки, а `y` содержит значения целевой переменной `cnt`.

Использование One-Hot Encoding для `hr` позволяет модели учитывать отдельные часы суток, не предполагая линейную зависимость между номерами часов.

## 5. Анализ мультиколлинеарности

Для оценки мультиколлинеарности используется коэффициент VIF — Variance Inflation Factor.

До удаления `atemp` были получены очень высокие значения VIF: примерно **42.24 для `atemp`** и **39.60 для `temp`**. Это связано с тем, что температура и ощущаемая температура являются очень близкими признаками.

Поэтому `atemp` был удалён, после чего VIF рассчитывается повторно.

In [ ]:
vif_data = pd.DataFrame()
vif_data['Признак'] = X.columns
vif_data['VIF'] = [
    variance_inflation_factor(X.values, i)
    for i in range(X.shape[1])
]

vif_data = vif_data.sort_values(by='VIF', ascending=False)

print('Показатели VIF:')
print(vif_data.to_string(index=False))

После удаления `atemp` максимальное значение VIF составляет примерно **2.21**. Все остальные значения также находятся на невысоком уровне и не превышают 5.

Таким образом, после удаления `atemp` выраженная мультиколлинеарность среди оставшихся признаков не наблюдается. Это является благоприятным результатом перед обучением регрессионных моделей.

## 6. Разделение данных и стандартизация

Данные разделяются на тренировочную и тестовую выборки в соотношении 80/20.

`X_train` и `y_train` используются для обучения моделей, а `X_test` и `y_test` — для итоговой проверки качества.

Для масштабирования признаков используется `StandardScaler`.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print('Размер X_train:', X_train.shape)
print('Размер X_test:', X_test.shape)
print('Размер y_train:', y_train.shape)
print('Размер y_test:', y_test.shape)

В тренировочной выборке находится **595 наблюдений**, а в тестовой — **149 наблюдений**. Такое разделение соответствует требуемой пропорции 80/20.

Стандартизация выполнена после разделения данных: параметры `StandardScaler` вычисляются на тренировочной выборке, после чего применяются к тестовой.

## 7. Базовые модели регрессии до PCA

Для решения задачи построены три модели:

- `LinearRegression` — обычная линейная регрессия;
- `Ridge(alpha=1.0)` — гребневая регрессия;
- `Lasso(alpha=0.1)` — Lasso-регрессия.

Для каждой модели рассчитываются RMSE, R² и MAPE.

In [ ]:
def evaluate_model(model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    mape = mean_absolute_percentage_error(y_test, y_pred) * 100

    print('RMSE:', rmse)
    print('R²:', r2)
    print('MAPE:', mape, '%')

    return model, y_pred

print('Линейная регрессия:')
linear_model = LinearRegression()
linear_model, y_pred = evaluate_model(
    linear_model,
    X_train_scaled,
    X_test_scaled,
    y_train,
    y_test
)

print('\nГребневая регрессия:')
ridge_model = Ridge(alpha=1.0)
ridge_model, ridge_y_pred = evaluate_model(
    ridge_model,
    X_train_scaled,
    X_test_scaled,
    y_train,
    y_test
)

print('\nLasso-регрессия:')
lasso_model = Lasso(alpha=0.1)
lasso_model, lasso_y_pred = evaluate_model(
    lasso_model,
    X_train_scaled,
    X_test_scaled,
    y_train,
    y_test
)

Для Linear Regression получено **RMSE ≈ 74.48**, **R² ≈ 0.750** и **MAPE ≈ 117.22%**. Значение R² показывает, что модель объясняет около 75% вариации целевой переменной на тестовой выборке.

Для Ridge получено **RMSE ≈ 74.54**, **R² ≈ 0.749** и **MAPE ≈ 117.00%**. Результаты практически совпадают с обычной линейной регрессией.

Для Lasso получено **RMSE ≈ 74.50**, **R² ≈ 0.750** и **MAPE ≈ 115.86%**. Качество также находится примерно на том же уровне.

Таким образом, до применения PCA различия между тремя моделями небольшие.

Высокое значение MAPE необходимо интерпретировать с осторожностью: в датасете встречаются часы с небольшим количеством аренд. При малом фактическом значении даже относительно небольшая абсолютная ошибка может давать большую процентную ошибку.

### 7.1. Кросс-валидация

Для дополнительной проверки устойчивости моделей используется 5-кратная кросс-валидация. В качестве показателя используется среднее значение R².

In [ ]:
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

models = {
    'Linear Regression': LinearRegression(),
    'Ridge': Ridge(alpha=1.0),
    'Lasso': Lasso(alpha=0.1)
}

cv_results = {}

for name, model in models.items():
    scores = cross_val_score(
        model,
        X_train_scaled,
        y_train,
        cv=kfold,
        scoring='r2'
    )
    cv_results[name] = scores.mean()
    print(name, '— средний CV R²:', scores.mean())

Среднее значение R² при 5-кратной кросс-валидации составило примерно **0.715 для Linear Regression**, **0.708 для Ridge** и **0.714 для Lasso**.

Значения немного ниже тестового R², но остаются положительными. Существенного различия между тремя моделями по результатам кросс-валидации также не наблюдается.

## 8. Метод главных компонент (PCA)

Перед применением PCA признаки были стандартизированы. В качестве критерия сохранения информации выбрано не менее 95% объяснённой дисперсии.

In [ ]:
pca = PCA(n_components=0.95)

X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

explained_variance = np.sum(pca.explained_variance_ratio_)

print('Результаты PCA:')
print('Количество исходных признаков:', X_train.shape[1])
print('Количество главных компонент:', X_train_pca.shape[1])
print('Объяснённая дисперсия:', explained_variance)

plt.figure(figsize=(8, 5))
plt.plot(
    np.cumsum(pca.explained_variance_ratio_),
    marker='o'
)
plt.axhline(y=0.95, linestyle='--')
plt.xlabel('Количество главных компонент')
plt.ylabel('Накопленная объяснённая дисперсия')
plt.title('Накопленная объяснённая дисперсия PCA')
plt.grid(alpha=0.3)
plt.show()

После применения PCA количество признаков уменьшилось с **35 до 30**, при этом сохранилось примерно **95.58% дисперсии**.

Таким образом, PCA действительно уменьшил размерность данных, однако уменьшение оказалось небольшим. Это соответствует результатам VIF: после удаления `atemp` выраженной мультиколлинеарности уже не наблюдалось.

## 9. Модели регрессии после PCA

После преобразования PCA повторно обучаются Linear Regression, Ridge и Lasso.

In [ ]:
print('Линейная регрессия после PCA:')
pca_linear_model = LinearRegression()
pca_linear_model, pca_y_pred = evaluate_model(
    pca_linear_model,
    X_train_pca,
    X_test_pca,
    y_train,
    y_test
)

print('\nГребневая регрессия после PCA:')
pca_ridge_model = Ridge(alpha=1.0)
pca_ridge_model, pca_ridge_y_pred = evaluate_model(
    pca_ridge_model,
    X_train_pca,
    X_test_pca,
    y_train,
    y_test
)

print('\nLasso-регрессия после PCA:')
pca_lasso_model = Lasso(alpha=0.1)
pca_lasso_model, pca_lasso_y_pred = evaluate_model(
    pca_lasso_model,
    X_train_pca,
    X_test_pca,
    y_train,
    y_test
)

После PCA для Linear Regression получено **RMSE ≈ 81.92**, **R² ≈ 0.697** и **MAPE ≈ 106.56%**.

Для Ridge получено **RMSE ≈ 81.94**, **R² ≈ 0.697** и **MAPE ≈ 106.62%**.

Для Lasso получено **RMSE ≈ 81.95**, **R² ≈ 0.697** и **MAPE ≈ 106.48%**.

По RMSE и R² после PCA наблюдается ухудшение качества относительно моделей на исходных признаках. При этом MAPE уменьшилась, однако эта метрика не даёт основания говорить об общем улучшении модели, поскольку разные метрики оценивают качество с разных сторон.

В рассматриваемой задаче PCA выполнил снижение размерности, но не улучшил качество прогноза.

## 10. Сравнение результатов до и после PCA

In [ ]:
results = {
    'Linear Regression': {
        'До PCA': [74.4760, 0.7499, 117.2158],
        'После PCA': [81.9199, 0.6974, 106.5589]
    },
    'Ridge': {
        'До PCA': [74.5421, 0.7495, 117.0032],
        'После PCA': [81.9353, 0.6973, 106.6214]
    },
    'Lasso': {
        'До PCA': [74.4998, 0.7498, 115.8606],
        'После PCA': [81.9543, 0.6972, 106.4837]
    }
}

print(f"{'Модель':<20} {'Период':<15} {'RMSE':<12} {'R²':<10} {'MAPE, %':<12}")
print('-' * 75)

for model_name, periods in results.items():
    for period, values in periods.items():
        rmse, r2, mape = values
        print(f'{model_name:<20} {period:<15} {rmse:<12.4f} {r2:<10.4f} {mape:<12.4f}')

До PCA все три модели показали близкие результаты: R² находится примерно на уровне **0.75**, а RMSE — около **74.5**.

После PCA RMSE увеличился примерно до **82**, а R² снизился примерно до **0.697**. Следовательно, в данной выборке снижение размерности привело к некоторой потере качества прогноза.

При этом PCA сохранил 95.58% дисперсии и сократил количество признаков с 35 до 30. Поэтому метод выполнил задачу снижения размерности, однако улучшения качества регрессии не произошло.

## 11. Анализ остатков

Остаток определяется как разность между фактическим и предсказанным значением:

`остаток = фактическое значение − предсказанное значение`.

Анализ остатков позволяет проверить характер ошибок модели и наличие неодинакового разброса ошибок.

In [ ]:
residuals = y_test - y_pred

print('Среднее значение остатков:', residuals.mean())
print('Стандартное отклонение остатков:', residuals.std())

plt.figure(figsize=(8, 5))
plt.scatter(y_pred, residuals)
plt.axhline(0, linestyle='--')
plt.title('Остатки в зависимости от предсказанных значений')
plt.xlabel('Предсказанные значения cnt')
plt.ylabel('Остатки')
plt.grid(alpha=0.3)
plt.show()

Среднее значение остатков составляет примерно **8.34**, что относительно невелико по сравнению со стандартным отклонением остатков **74.26**. Поэтому сильного систематического смещения среднего остатка относительно нуля не наблюдается.

На графике остатки расположены по обе стороны от нулевой линии, однако при увеличении предсказанных значений их разброс становится больше. Это может свидетельствовать о неодинаковой дисперсии ошибок.

### 11.1. Тест Бреуша — Пагана

Для дополнительной проверки постоянства дисперсии ошибок используется тест Бреуша — Пагана.

In [ ]:
X_test_with_constant = sm.add_constant(X_test_scaled)
test_results = het_breuschpagan(residuals, X_test_with_constant)

print('LM-статистика:', test_results[0])
print('p-value:', test_results[1])
print('F-статистика:', test_results[2])
print('F p-value:', test_results[3])

Для теста Бреуша — Пагана получено **p-value ≈ 4.04 × 10⁻⁵**. Это значение меньше 0.05.

Следовательно, гипотеза о постоянной дисперсии остатков отвергается. Результат теста подтверждает наличие **гетероскедастичности**: разброс ошибок зависит от уровня предсказанного значения.

Этот результат следует учитывать при интерпретации качества обычной линейной регрессии.

## 12. Заключение

В лабораторной работе был исследован датасет Bike Sharing Dataset и построены модели для прогнозирования почасового количества аренд велосипедов `cnt`.

Для исследования был выбран май 2011 года, содержащий 744 наблюдения. В ходе подготовки данных были удалены признаки, которые не использовались в модели, а также `casual` и `registered`, чтобы избежать утечки целевой переменной.

Корреляционный анализ показал умеренную обратную связь количества аренд с влажностью и прямую связь с температурой. Отсутствие очень высокой корреляции не означает отсутствия зависимости, поскольку часть зависимостей, например зависимость от часа суток, может быть нелинейной.

При анализе VIF была обнаружена сильная мультиколлинеарность между `temp` и `atemp`. После удаления `atemp` максимальное значение VIF снизилось примерно до 2.21, поэтому выраженной мультиколлинеарности среди оставшихся признаков не наблюдается.

До PCA Linear Regression, Ridge и Lasso показали близкие результаты: R² около 0.75 и RMSE около 74.5.

PCA уменьшил количество признаков с 35 до 30 и сохранил около 95.58% дисперсии. Однако после PCA RMSE увеличился примерно до 82, а R² снизился примерно до 0.697. Поэтому в данной задаче PCA не улучшил качество прогноза.

Анализ остатков показал наличие неодинакового разброса ошибок. Тест Бреуша — Пагана подтвердил наличие гетероскедастичности, так как p-value оказалось меньше 0.05.

Таким образом, основные задачи лабораторной работы выполнены: данные подготовлены и исследованы, проведены корреляционный анализ и расчёт VIF, построены три модели регрессии, выполнена кросс-валидация, применён PCA и проведено сравнение моделей до и после снижения размерности.